# Decision Tree Classifier from Scratch
This notebook implements a decision tree classifier using Gini Impurity, including visualization.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Constants
COUNT_THRESHOLD = 1
MAX_DEPTH = 4


In [ ]:

# Gini Impurity
def calculate_gini(data):
    total = len(data)
    if total == 0:
        return 0
    label_counts = {}
    for row in data:
        label = row[-1]
        label_counts[label] = label_counts.get(label, 0) + 1
    gini = 1.0
    for count in label_counts.values():
        prob = count / total
        gini -= prob ** 2
    return gini


In [ ]:

# Split data based on attribute
def split_data(data, attribute_index, value):
    return [row for row in data if row[attribute_index] == value]


In [ ]:

# Gini Gain
def gini_gain(data, attribute_index):
    parent_gini = calculate_gini(data)
    total = len(data)
    values = set(row[attribute_index] for row in data)
    weighted_gini = 0.0
    for value in values:
        subset = split_data(data, attribute_index, value)
        weighted_gini += (len(subset) / total) * calculate_gini(subset)
    return parent_gini - weighted_gini


In [ ]:

# Majority class
def majority_label(data):
    label_counts = {}
    for row in data:
        label = row[-1]
        label_counts[label] = label_counts.get(label, 0) + 1
    return max(label_counts.items(), key=lambda x: x[1])[0]


In [ ]:

# Build the tree
def build_tree(data, attributes, depth=0):
    labels = [row[-1] for row in data]
    if labels.count(labels[0]) == len(labels):  # pure node
        return labels[0]
    if len(data) < COUNT_THRESHOLD or depth >= MAX_DEPTH or not attributes:
        return majority_label(data)

    best_attr = max(attributes, key=lambda attr: gini_gain(data, attr))
    tree = {best_attr: {}}
    values = set(row[best_attr] for row in data)
    for value in values:
        subset = split_data(data, best_attr, value)
        if not subset:
            tree[best_attr][value] = majority_label(data)
        else:
            tree[best_attr][value] = build_tree(subset, [a for a in attributes if a != best_attr], depth + 1)
    return tree


In [ ]:

# Prediction
def predict(tree, instance):
    while isinstance(tree, dict):
        attr = next(iter(tree))
        val = instance.get(attr)
        if val not in tree[attr]:
            return None
        tree = tree[attr][val]
    return tree


In [ ]:

# Tree plotting with improved spacing
def plot_tree(tree, depth=0, x_offset=0.5, y_offset=1.0, x_gap=0.25, ax=None, data=None, attribute_names=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.axis('off')

    if not isinstance(tree, dict):
        if data:
            label_counts = {label: sum(1 for row in data if row[-1] == label) for label in set(row[-1] for row in data)}
            label_text = f"{tree}\n" + "\n".join([f"{label}: {count}" for label, count in label_counts.items()])
        else:
            label_text = str(tree)
        ax.text(x_offset, y_offset, label_text, fontsize=10, ha='center', 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", edgecolor="black"))
        return

    root = next(iter(tree))
    attr_name = attribute_names[root] if attribute_names else str(root)
    ax.text(x_offset, y_offset, attr_name, fontsize=12, ha='center',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", edgecolor="black"))

    children = list(tree[root].items())
    num_children = len(children)
    for i, (value, subtree) in enumerate(children):
        child_x = x_offset + (i - (num_children - 1) / 2) * x_gap * (depth + 1)
        child_y = y_offset - 0.15
        ax.plot([x_offset, child_x], [y_offset - 0.02, child_y + 0.02], 'k-', lw=1)
        ax.text((x_offset + child_x) / 2, (y_offset + child_y) / 2 + 0.01, str(value),
                ha='center', fontsize=9, color='gray')

        subset = split_data(data, root, value)
        plot_tree(subtree, depth + 1, child_x, child_y, x_gap, ax=ax, data=subset, attribute_names=attribute_names)

    if depth == 0:
        plt.show()


In [ ]:

# Sample data and usage
data = [
    ['Sunny', 'Hot', 'High', 'No'],
    ['Sunny', 'Hot', 'High', 'No'],
    ['Overcast', 'Hot', 'High', 'Yes'],
    ['Rain', 'Mild', 'High', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Yes'],
    ['Sunny', 'Mild', 'High', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Yes'],
]

attribute_names = ['Outlook', 'Temperature', 'Humidity']
attributes = list(range(len(attribute_names)))

tree = build_tree(data, attributes)
plot_tree(tree, data=data, attribute_names=attribute_names)

# Predict a sample instance
instance = {0: 'Rain', 1: 'Mild', 2: 'High'}
print("Predicted:", predict(tree, instance))
